# Analysis

Sentiment, categorization, clustering, and Plotly exploration for the discovery
records. The same pipeline is applied to both **challenges** and **expectations**.

Section 1 runs `run()` to build `output/processed/challenges.csv` and
`expectations.csv` from the source notes/worksheets (no terminal step needed).

In [ ]:
from pathlib import Path

def _requirements() -> Path:
    """Find simple/requirements.txt whether kernel cwd is simple/ or notebook/."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        req = candidate / "requirements.txt"
        if req.is_file() and (candidate / "scripts").is_dir():
            return req
    raise FileNotFoundError(
        "Could not find requirements.txt next to scripts/. "
        f"cwd={here}"
    )

%pip install -q -r {_requirements()}
%pip install -q "nbformat>=4.2.0"


: 

# 0. Import Libraries

In [ ]:
from pathlib import Path
import os
import sys

import hdbscan
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import umap
from IPython.display import Markdown, display
from sklearn.metrics import silhouette_score

# Resolve project root (simple/) whether the kernel cwd is simple/ or notebook/
def _project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "scripts" / "sentiment_analysis.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find project root (expected scripts/sentiment_analysis.py). "
        f"cwd={here}"
    )


ROOT = _project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.categorize_records import CATEGORY_CONFIG, CATEGORY_DESCRIPTIONS, run_categorize
from scripts.extract_records import SOURCE_MEETING_NOTES, load_prepared_records, run
from scripts.sentiment_analysis import realign_by_sentiment, run_sentiment

PROCESSED_DIR = ROOT / "output" / "processed"
CHALLENGES_CSV = PROCESSED_DIR / "challenges.csv"
EXPECTATIONS_CSV = PROCESSED_DIR / "expectations.csv"

CHALLENGES_SCORED = PROCESSED_DIR / "challenges_scored.csv"
EXPECTATIONS_SCORED = PROCESSED_DIR / "expectations_scored.csv"
CATEGORIZED_CHALLENGES_CSV = PROCESSED_DIR / "categorized_challenges.csv"
CATEGORIZED_EXPECTATIONS_CSV = PROCESSED_DIR / "categorized_expectations.csv"
CATEGORY_SUMMARY_CSV = PROCESSED_DIR / "category_summary.csv"
CATEGORY_SUMMARY_XLSX = PROCESSED_DIR / "category_summary.xlsx"

CHALLENGE_TEXT_COL = "pain_points"
EXPECTATION_TEXT_COL = "expectations"


## 1. Extract, load & prep records

Runs `run()` to extract from source notes/worksheets into CSVs, then `load_prepared_records()` (focus-group aliases, source tags, short meeting-note merge).


In [ ]:
run()  # writes output/processed/challenges.csv and expectations.csv
challenges, expectations = load_prepared_records(CHALLENGES_CSV, EXPECTATIONS_CSV)

print(f"Challenges: {len(challenges)} | Expectations: {len(expectations)}")
print(challenges["source"].value_counts().to_string())

In [ ]:
challenges.sample(10)

## 2. Sentiment Analysis

Uses `Thi144/sentiment-distilbert-7class`, mapped to negative / neutral / positive.


In [ ]:
challenges, expectations = run_sentiment(challenges, expectations)

print("Challenge sentiment:\n", challenges["sentiment"].value_counts().to_string())
print("\nExpectation sentiment:\n", expectations["sentiment"].value_counts().to_string())

challenges.to_csv(CHALLENGES_SCORED, index=False, encoding="utf-8-sig")
expectations.to_csv(EXPECTATIONS_SCORED, index=False, encoding="utf-8-sig")
print(f"Saved → {CHALLENGES_SCORED.name}, {EXPECTATIONS_SCORED.name}")

## 3. Realign misclassified rows

Move positive “pain” → expectations and negative “expectations” → pain, **meeting_notes only**.


In [ ]:
# Positive challenges from meeting_notes (candidates to move to expectations)
challenges.loc[
    (challenges["sentiment"] == "positive")
    & (challenges["source"] == SOURCE_MEETING_NOTES)
]

In [ ]:
n_pos = int(
    ((challenges["source"] == SOURCE_MEETING_NOTES) & (challenges["sentiment"] == "positive")).sum()
)
n_neg = int(
    ((expectations["source"] == SOURCE_MEETING_NOTES) & (expectations["sentiment"] == "negative")).sum()
)

challenges, expectations = realign_by_sentiment(
    challenges, expectations, only_source=SOURCE_MEETING_NOTES
)

print(f"Moved meeting_notes only: {n_pos} positive→expectations, {n_neg} negative→challenges")
print(f"Shapes: challenges {len(challenges)}, expectations {len(expectations)}")
print(challenges["source"].value_counts().to_string())

## 3b. Validate sentiment after realignment

Confirm meeting-note positives no longer sit in challenges and negatives no longer sit in expectations, then chart the post-realign mix.


In [ ]:
CORAL = "#E76F51"
TEAL = "#2A9D8F"
MUTED = "#94A3B8"
NAVY = "#1B3A4B"

order = ["negative", "neutral", "positive"]
colors = {"negative": CORAL, "neutral": MUTED, "positive": TEAL}


def counts(frame: pd.DataFrame) -> list[int]:
    vc = frame["sentiment"].value_counts()
    return [int(vc.get(s, 0)) for s in order]


def pct(vals: list[int]) -> list[float]:
    total = sum(vals) or 1
    return [100 * v / total for v in vals]


# --- Validation: realign should clear these meeting_notes mismatches ---
pos_in_challenges = challenges.loc[
    (challenges["source"] == SOURCE_MEETING_NOTES)
    & (challenges["sentiment"] == "positive")
]
neg_in_expectations = expectations.loc[
    (expectations["source"] == SOURCE_MEETING_NOTES)
    & (expectations["sentiment"] == "negative")
]

print("Post-realign validation (meeting_notes only)")
print(f"  Positive challenges remaining: {len(pos_in_challenges)} (expect 0)")
print(f"  Negative expectations remaining: {len(neg_in_expectations)} (expect 0)")
if len(pos_in_challenges) or len(neg_in_expectations):
    display(Markdown("**Still misfiled after realign — inspect below**"))
    if len(pos_in_challenges):
        display(pos_in_challenges)
    if len(neg_in_expectations):
        display(neg_in_expectations)
else:
    print("  OK — no meeting_notes polarity mismatches left in the wrong table.")

print("\nSentiment counts after realign:")
print("  Challenges:", dict(zip(order, counts(challenges))))
print("  Expectations:", dict(zip(order, counts(expectations))))

ch, ex = counts(challenges), counts(expectations)
x = np.arange(len(order))
w = 0.36

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(x - w / 2, ch, width=w, color=CORAL, label="Challenges")
axes[0].bar(x + w / 2, ex, width=w, color=TEAL, label="Expectations")
axes[0].set_xticks(x)
axes[0].set_xticklabels([s.title() for s in order])
axes[0].set_ylabel("Records")
axes[0].set_title("Sentiment counts (post-realign)")
axes[0].legend(frameon=False, fontsize=8)
axes[0].set_axisbelow(True)
axes[0].yaxis.grid(True, color="#E2E8F0", lw=0.8)

ch_p, ex_p = pct(ch), pct(ex)
bottom = np.zeros(2)
for i, sent in enumerate(order):
    vals = [ch_p[i], ex_p[i]]
    axes[1].bar(
        ["Challenges", "Expectations"],
        vals,
        bottom=bottom,
        color=colors[sent],
        label=sent.title(),
        width=0.55,
    )
    bottom = bottom + vals
axes[1].set_ylim(0, 100)
axes[1].set_ylabel("Percent")
axes[1].set_title("Sentiment mix within each table")
axes[1].legend(frameon=False, fontsize=8, loc="upper right")

fig.suptitle(
    "Sentiment signal after realignment",
    fontsize=13,
    fontweight="bold",
    color=NAVY,
    y=1.02,
)
fig.tight_layout()
fig_path = ROOT / "output" / "figures" / "sentiment_mix_post_realign.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_path, dpi=180, bbox_inches="tight", facecolor="white")
print(f"Saved {fig_path}")
plt.show()


## 4. Categorize records

Hybrid: body keywords → semantic similarity.


In [ ]:
challenges, expectations, challenge_embeddings, embedder = run_categorize(
    challenges, expectations
)

# run_categorize only returns challenge embeddings; embed expectations too (reusing the
# categorize_text column it added) so they can be clustered the same way.
expectation_embeddings = embedder.encode(
    expectations["categorize_text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64,
)

print("Challenge categories:\n", challenges["Category"].value_counts().to_string())
print("\nExpectation categories:\n", expectations["Category"].value_counts().to_string())

## 5. Cluster challenges & expectations (UMAP + HDBSCAN)

In [ ]:
def assign_cluster_names(
    frame: pd.DataFrame,
    cluster_col: str = "Cluster",
    category_col: str = "Category",
    output_col: str = "Cluster_Label",
) -> tuple[pd.DataFrame, pd.DataFrame]:
    cluster_summary = (
        frame.groupby([cluster_col, category_col]).size().reset_index(name="Count")
    )
    dominant = (
        cluster_summary.sort_values("Count", ascending=False)
        .drop_duplicates(subset=[cluster_col])
        [[cluster_col, category_col]]
        .rename(columns={category_col: output_col})
    )
    return frame.merge(dominant, on=cluster_col, how="left"), cluster_summary


def soft_assign_noise(
    coords: np.ndarray,
    labels: np.ndarray,
    radius_percentile: float = 92,
    radius_slack: float = 1.35,
) -> tuple[np.ndarray, int]:
    """Pull leftover noise into the nearest cluster if it sits inside that cluster's envelope."""
    out = labels.copy()
    clustered = out != -1
    if not clustered.any():
        return out, 0

    centroids, radii = {}, {}
    for cid in sorted(set(out[clustered])):
        pts = coords[out == cid]
        centroids[cid] = pts.mean(axis=0)
        dists = np.linalg.norm(pts - centroids[cid], axis=1)
        radii[cid] = float(np.percentile(dists, radius_percentile) * radius_slack)

    assigned = 0
    for i in np.where(out == -1)[0]:
        best_c, best_d = None, np.inf
        for cid, center in centroids.items():
            d = float(np.linalg.norm(coords[i] - center))
            if d < best_d:
                best_c, best_d = cid, d
        if best_c is not None and best_d <= radii[best_c]:
            out[i] = best_c
            assigned += 1
    return out, assigned


TARGET_CLUSTERS = len(CATEGORY_CONFIG)
# Prefer coverage over ultra-tight cores (prior sweep left ~36% as noise).
NOISE_PENALTY = 1.35
CLUSTER_COUNT_PENALTY = 0.012


def cluster_frame(frame, embeddings, label):
    """UMAP + HDBSCAN param sweep, soft-assign noise, then name clusters by majority category."""
    reduced = umap.UMAP(
        n_neighbors=15, n_components=12, min_dist=0.0, metric="cosine", random_state=42
    ).fit_transform(embeddings)

    candidate_params = sorted({
        (mcs, ms, method)
        for mcs in range(5, 14)
        for ms in (1, 2, 3, max(1, mcs // 3))
        for method in ("eom", "leaf")
    })

    sweep_rows, best_labels, best_params, best_score = [], None, None, -np.inf
    n_rows = max(len(frame), 1)
    for mcs, ms, method in candidate_params:
        labels = hdbscan.HDBSCAN(
            min_cluster_size=mcs,
            min_samples=ms,
            metric="euclidean",
            cluster_selection_method=method,
        ).fit_predict(reduced)

        mask = labels != -1
        n_clusters = len(set(labels[mask]))
        noise = int((labels == -1).sum())
        noise_frac = noise / n_rows
        silhouette = (
            silhouette_score(reduced[mask], labels[mask], metric="euclidean")
            if mask.sum() > 1 and n_clusters > 1
            else -1.0
        )
        if noise_frac > 0.45 or n_clusters < 4 or n_clusters > 40:
            combined = -1.0
        else:
            combined = (
                silhouette
                - abs(n_clusters - TARGET_CLUSTERS) * CLUSTER_COUNT_PENALTY
                - noise_frac * NOISE_PENALTY
            )
        sweep_rows.append({
            "min_cluster_size": mcs, "min_samples": ms, "method": method,
            "clusters": n_clusters, "noise": noise, "noise_pct": round(noise_frac * 100, 1),
            "silhouette": round(float(silhouette), 4), "combined_score": round(float(combined), 4),
        })
        if combined > best_score:
            best_score, best_labels, best_params = combined, labels, (mcs, ms, method)

    # Fallback if every candidate was rejected (e.g. a small frame).
    if best_labels is None:
        best_params = (5, 1, "eom")
        best_labels = hdbscan.HDBSCAN(
            min_cluster_size=5, min_samples=1, metric="euclidean",
            cluster_selection_method="eom",
        ).fit_predict(reduced)

    print(f"[{label}] Best HDBSCAN params: {best_params} score: {round(float(best_score), 4)}")
    display(pd.DataFrame(sweep_rows).sort_values("combined_score", ascending=False).head(5))

    raw_labels = best_labels.copy()
    raw_noise = int((raw_labels == -1).sum())
    best_labels, n_soft = soft_assign_noise(reduced, best_labels)
    print(
        f"[{label}] Soft-assigned {n_soft} noise points "
        f"(noise {raw_noise} → {int((best_labels == -1).sum())})"
    )

    frame = frame.copy()
    frame["Cluster_Raw"] = raw_labels
    frame["Cluster"] = best_labels
    frame["Assign_Status"] = np.where(
        raw_labels != -1,
        "HDBSCAN cluster",
        np.where(best_labels != -1, "Soft-assigned", "Unclustered"),
    )
    frame, summary = assign_cluster_names(frame)
    frame.loc[frame["Cluster"] == -1, "Cluster_Label"] = "Noise / unclustered"

    print(f"[{label}] Assignment status:")
    print(frame["Assign_Status"].value_counts().to_string())
    print(
        f"[{label}] Unclustered: {(frame['Cluster'] == -1).sum()} / {len(frame)} "
        f"({100 * (frame['Cluster'] == -1).mean():.1f}%)\n"
    )
    return frame, reduced, summary


challenges, challenge_reduced, challenge_cluster_summary = cluster_frame(
    challenges, challenge_embeddings, "challenges"
)
expectations, expectation_reduced, expectation_cluster_summary = cluster_frame(
    expectations, expectation_embeddings, "expectations"
)

### Soft assignment map

2D projection of the same UMAP space used for HDBSCAN, for challenges and
expectations. Color by assignment status to see which points were dense cores,
which were rescued by soft assignment, and which stayed unclustered.

In [ ]:
def soft_assignment_map(frame, embeddings, text_col, title):
    coords = umap.UMAP(
        n_neighbors=15, n_components=2, min_dist=0.1, metric="cosine", random_state=42
    ).fit_transform(embeddings)
    plot = pd.DataFrame({
        "x": coords[:, 0],
        "y": coords[:, 1],
        "Assign_Status": frame["Assign_Status"].values,
        "Cluster_Label": frame["Cluster_Label"].values,
        "Category": frame["Category"].values,
        "Focus Group": frame["focus_group"].values,
        "Record": frame[text_col].values,
    })
    fig = px.scatter(
        plot, x="x", y="y", color="Assign_Status",
        hover_data=["Focus Group", "Cluster_Label", "Category", "Record"],
        title=title,
        color_discrete_map={
            "HDBSCAN cluster": "#2A9D8F", "Soft-assigned": "#E9C46A", "Unclustered": "#E76F51",
        },
    )
    fig.update_layout(height=560, width=1100, template="plotly_white")
    fig.show()


soft_assignment_map(challenges, challenge_embeddings, CHALLENGE_TEXT_COL,
                    "Challenges — soft assignment map (UMAP 2D)")
soft_assignment_map(expectations, expectation_embeddings, EXPECTATION_TEXT_COL,
                    "Expectations — soft assignment map (UMAP 2D)")

### Cluster contents

Cluster sizes for challenges and expectations, including noise (`Cluster == -1`).

In [ ]:
def cluster_sizes(frame, text_col, label):
    cols = [
        c for c in [
            "Cluster", "Cluster_Label", "Assign_Status", "focus_group",
            text_col, "Category", "source",
        ] if c in frame.columns
    ]
    show = frame[cols].sort_values(["Cluster", "Assign_Status", "focus_group"]).reset_index(drop=True)
    print(f"[{label}] Cluster sizes (including unclustered):")
    display(
        show.groupby(["Cluster", "Cluster_Label"], as_index=False)
        .size()
        .rename(columns={"size": "Count"})
        .sort_values("Cluster")
    )


cluster_sizes(challenges, CHALLENGE_TEXT_COL, "challenges")
cluster_sizes(expectations, EXPECTATION_TEXT_COL, "expectations")

## 6. Visualizations


In [ ]:
def semantic_map(frame, embeddings, text_col, title):
    coords = umap.UMAP(
        n_neighbors=15, n_components=2, min_dist=0.1, metric="cosine", random_state=42
    ).fit_transform(embeddings)
    plot = pd.DataFrame({
        "x": coords[:, 0],
        "y": coords[:, 1],
        "Category": frame["Category"].values,
        "Cluster_Label": frame["Cluster_Label"].values,
        "Record": frame[text_col].values,
        "Focus Group": frame["focus_group"].values,
    })
    fig = px.scatter(
        plot, x="x", y="y", color="Category",
        hover_data=["Focus Group", "Cluster_Label", "Record"], title=title,
    )
    fig.update_layout(height=560, width=1100, template="plotly_white")
    fig.show()


semantic_map(challenges, challenge_embeddings, CHALLENGE_TEXT_COL,
             "Challenges — 2D semantic map by category")
semantic_map(expectations, expectation_embeddings, EXPECTATION_TEXT_COL,
             "Expectations — 2D semantic map by category")

In [ ]:
from collections import Counter
from wordcloud import WordCloud


def word_cloud(text_series, title, min_count=1):
    words = " ".join(text_series.astype(str).values).lower().split()
    freqs = Counter(words)
    if min_count > 1:
        freqs = {w: c for w, c in freqs.items() if c >= min_count}
    cloud = WordCloud(width=6800, height=2400, background_color="white").generate_from_frequencies(freqs)
    plt.figure(figsize=(10, 5))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(title)
    plt.show()


# Challenge words occurring 3+ times
word_cloud(challenges[CHALLENGE_TEXT_COL], "Challenges word cloud", min_count=3)

In [ ]:
# Expectation words occurring 2+ times
word_cloud(expectations[EXPECTATION_TEXT_COL], "Expectations word cloud", min_count=2)

## 6b. Reconcile categories → `final_category`

**Cluster purity** = share of a cluster whose per-record `Category` matches the
cluster's dominant label (`Cluster_Label`). `final_category` keeps the record's own
category when it is confident, but adopts the cluster label when the record is
low-confidence *and* lives in a pure cluster.

In [ ]:
def add_final_category(frame, purity_threshold=0.6, low_conf=3.0):
    """Blend per-record Category with the cluster's dominant label (Cluster_Label)."""
    frame = frame.copy()
    match = frame["Category"] == frame["Cluster_Label"]
    purity = match.groupby(frame["Cluster"]).transform("mean")
    frame["Cluster_Purity"] = np.where(frame["Cluster"] == -1, np.nan, purity)

    frame["final_category"] = frame["Category"]
    take_cluster = (
        (frame["Cluster"] != -1)
        & (frame["Cluster_Purity"] >= purity_threshold)
        & (frame["Category_Confidence"] < low_conf)
    )
    frame.loc[take_cluster, "final_category"] = frame.loc[take_cluster, "Cluster_Label"]
    frame["Category_Reassigned"] = frame["final_category"] != frame["Category"]
    return frame


challenges = add_final_category(challenges)
expectations = add_final_category(expectations)

for label, frame in [("challenges", challenges), ("expectations", expectations)]:
    n = int(frame["Category_Reassigned"].sum())
    print(f"{label}: {n}/{len(frame)} records took the cluster label "
          f"({100 * n / len(frame):.1f}%)")

## 7. Save categorized outputs


In [ ]:
record_cols = [
    "department", "focus_group", "source", "sentiment",
    "Category", "Category_Method", "Category_Confidence",
    "Cluster_Label", "Cluster", "Cluster_Purity", "Assign_Status",
    "final_category",
]
challenge_cols = [c for c in ([CHALLENGE_TEXT_COL] + record_cols) if c in challenges.columns]
expectation_cols = [c for c in ([EXPECTATION_TEXT_COL] + record_cols) if c in expectations.columns]

challenges[challenge_cols].to_csv(CATEGORIZED_CHALLENGES_CSV, index=False, encoding="utf-8-sig")
expectations[expectation_cols].to_csv(CATEGORIZED_EXPECTATIONS_CSV, index=False, encoding="utf-8-sig")


def category_summary(frame, category_col="final_category"):
    return (
        frame.groupby(category_col, as_index=False).size()
        .rename(columns={"size": "Count", category_col: "Category"})
        .sort_values("Count", ascending=False)
    )


challenge_summary = category_summary(challenges)
expectation_summary = category_summary(expectations)
challenge_summary.to_csv(CATEGORY_SUMMARY_CSV, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(CATEGORY_SUMMARY_XLSX, engine="openpyxl") as writer:
    challenge_summary.to_excel(writer, sheet_name="challenge_summary", index=False)
    expectation_summary.to_excel(writer, sheet_name="expectation_summary", index=False)
    challenges[challenge_cols].to_excel(writer, sheet_name="categorized_challenges", index=False)
    expectations[expectation_cols].to_excel(writer, sheet_name="categorized_expectations", index=False)

print(f"Saved {CATEGORIZED_CHALLENGES_CSV} ({len(challenges)} rows)")
print(f"Saved {CATEGORIZED_EXPECTATIONS_CSV} ({len(expectations)} rows)")
print(f"Saved {CATEGORY_SUMMARY_CSV}")
print(f"Saved {CATEGORY_SUMMARY_XLSX} (sheets: challenge_summary, expectation_summary, "
      "categorized_challenges, categorized_expectations)")

print("\nChallenge final categories:")
display(challenge_summary)
print("Expectation final categories:")
display(expectation_summary)